In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

In [2]:
fusion = pd.read_csv(
    "../dataset/processed/adaptive_fusion_dataset.csv"
)

print(fusion.shape)

fusion.head()

(45493, 36)


,Traffic_Score,Workload,Multiple_Deliveries,Peak,Festival,Rider_Experience,Ratings,Traffic_Workload,Demand_Index,Rider_Load,...,Experience,Delivery_Index,Weather_Impact,Experience_Index,Order_Hour,Pickup_Hour,Weekend,Month,Delivery_person_Age,Time_taken (min)
0,4,3.0,3.0,3,0,151.2,4.2,12.0,16,37.80,...,151.2,41.122328,41.122328,30.240000,21,22,1,2,36.0,46
1,3,1.0,1.0,2,0,98.7,4.7,3.0,9,49.35,...,98.7,18.726956,31.211593,24.675000,14,15,1,2,21.0,23
2,2,1.0,1.0,0,0,108.1,4.7,2.0,2,54.05,...,108.1,27.575720,82.727161,36.033333,17,17,0,3,23.0,21
3,1,0.0,0.0,1,0,146.2,4.3,0.0,2,146.20,...,146.2,2.930258,17.581547,73.100000,9,9,1,2,34.0,20
4,4,1.0,1.0,3,0,112.8,4.7,4.0,16,56.40,...,112.8,77.586473,77.586473,22.560000,19,20,0,2,24.0,41


In [3]:
TARGET = "Time_taken (min)"

X = fusion.drop(columns=[TARGET])

y = fusion[TARGET]

print(X.shape)

(45493, 35)


In [4]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical")

print(categorical_features)

print()

print("Numerical")

print(numerical_features)

Categorical
[]

Numerical
['Traffic_Score', 'Workload', 'Multiple_Deliveries', 'Peak', 'Festival', 'Rider_Experience', 'Ratings', 'Traffic_Workload', 'Demand_Index', 'Rider_Load', 'Trip_Distance', 'Traffic', 'Vehicle', 'Vehicle_Condition', 'Restaurant_Lat', 'Restaurant_Lon', 'Travel_Index', 'Vehicle_Index', 'Efficiency', 'Weather', 'City', 'Order', 'Restaurant_Demand', 'Weather_Delay', 'Lat', 'Lon', 'Experience', 'Delivery_Index', 'Weather_Impact', 'Experience_Index', 'Order_Hour', 'Pickup_Hour', 'Weekend', 'Month', 'Delivery_person_Age']


In [ ]:
numeric_transformer = Pipeline(

    steps=[

        ("imputer",
         SimpleImputer(strategy="median"))

    ]

)

categorical_transformer = Pipeline(

    steps=[

        ("imputer",
         SimpleImputer(strategy="most_frequent")),

        ("encoder",
         OneHotEncoder(handle_unknown="ignore"))

    ]

)

preprocessor = ColumnTransformer(

    transformers=[

        (

            "num",
            numeric_transformer,
            numerical_features
        ),

        (
            "cat",
            categorical_transformer,
            categorical_features

        )

    ]

)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X,y, test_size=0.20,    random_state=42

)

print(X_train.shape)

print(X_test.shape)

(36394, 35)
(9099, 35)


In [7]:
eta_engine = Pipeline(

    steps=[

        (

            "preprocessor",

            preprocessor

        ),

        (

            "model",

            XGBRegressor(

                n_estimators=600,

                learning_rate=0.03,

                max_depth=8,

                subsample=0.85,

                colsample_bytree=0.85,

                objective="reg:squarederror",

                random_state=42,

                n_jobs=-1

            )

        )

    ]

)

In [8]:
eta_engine.fit(

    X_train,

    y_train

)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Traffic_Score', 'Workload',
                                                   'Multiple_Deliveries',
                                                   'Peak', 'Festival',
                                                   'Rider_Experience',
                                                   'Ratings',
                                                   'Traffic_Workload',
                                                   'Demand_Index', 'Rider_Load',
                                                   'Trip_Distance', 'Traffic',
                                                   'Vehicle',
                                                   'Vehicle_Condition',
                                                   'Restaurant_Lat'...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.03,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=8, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=600, n_jobs=-1,
                              num_parallel_tree=None, random_state=42, ...))])

In [9]:
pred = eta_engine.predict(

    X_test
)

In [10]:
mae = mean_absolute_error(

    y_test,

    pred

)

rmse = np.sqrt(

    mean_squared_error(

        y_test,

        pred

    )

)

r2 = r2_score(

    y_test,

    pred

)

print("="*50)

print("Adaptive ETA Engine Performance")

print("="*50)

print(f"MAE : {mae:.3f}")

print(f"RMSE: {rmse:.3f}")

print(f"R²  : {r2:.4f}")

Adaptive ETA Engine Performance
MAE : 3.132
RMSE: 3.942
R²  : 0.8242


In [11]:
joblib.dump(

    eta_engine,

    "../ModelV3/adaptive_eta_engine.pkl"

)

print("Adaptive ETA Engine Saved")

Adaptive ETA Engine Saved


In [14]:
results = pd.DataFrame({

    "Actual":y_test,

    "Predicted":pred

})

results.to_csv(

    "../dataset/processed/adaptive_predictions.csv",

    index=False

)

results.head()

,Actual,Predicted
6108,28,30.961443
40056,44,35.216942
27648,32,32.612194
33295,27,28.581631
31545,10,15.423270


In [15]:
comparison = pd.DataFrame({

    "Architecture":[

        "Linear Regression",

        "Random Forest",

        "XGBoost",

        "Adaptive ETA Engine"

    ],

    "MAE":[

        4.77,

        3.10,

        3.05,

        mae

    ],

    "RMSE":[

        6.00,

        3.94,

        3.83,

        rmse

    ],

    "R2":[

        0.5927,

        0.8243,

        0.8336,

        r2

    ]

})

comparison

,Architecture,MAE,RMSE,R2
0,Linear Regression,4.77000,6.000000,0.592700
1,Random Forest,3.10000,3.940000,0.824300
2,XGBoost,3.05000,3.830000,0.833600
3,Adaptive ETA Engine,3.13199,3.941562,0.824162
